In [1]:
import sqlite3
import pandas as pd
import pantab
from pathlib import Path

folder = Path(r"C:\Users\alrazz\Documents\Anonating files")


In [2]:
db_paths = list(folder.glob("*.db"))
#print (db_paths)

def load_texts_from_dbs(db_paths):
    dfs = {}

    for db_path in db_paths:
        with sqlite3.connect(db_path) as conn:
            dfs[db_path.stem] = pd.read_sql_query(
                'SELECT u, id, ts, text, TURKU_NLP, TURKU_NLP_sub, Tags, "web-register" FROM texts;',
                conn
            )

    return dfs

dfs = load_texts_from_dbs(db_paths)

#dfs["HI_clean_dedup"].head()

In [3]:
print(dfs["HI_clean_dedup"].columns)

Index(['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags',
       'web-register'],
      dtype='object')


In [4]:
dfs_noNA = {
    name: df.dropna(subset=["Turku_NLP", "Turku_NLP_sub"])
    for name, df in dfs.items()
}

for name, df in dfs_noNA.items():
    print(name, df.shape)

ed (47, 8)
ed_2 (98, 8)
ed_3 (5, 8)
en (3, 8)
en_10 (17, 8)
en_11 (19, 8)
en_2 (3, 8)
en_3 (11, 8)
en_4 (8, 8)
en_5 (7, 8)
en_6 (16, 8)
en_7 (4, 8)
en_8 (4, 8)
en_9 (24, 8)
fi (147, 8)
HI_2_clean_dedup (85, 8)
HI_clean_dedup (61, 8)
HI_ID_LY_SP_clean2_dedup (6, 8)
ID_clean_dedup (225, 8)
it (6, 8)
it_2 (7, 8)
it_3 (91, 8)
lt (149, 8)
lt_2 (58, 8)
lt_3 (75, 8)
lt_4 (30, 8)
LY_clean_dedup (181, 8)
MT (148, 8)
nb (15, 8)
nb_2 (100, 8)
nb_3 (0, 8)
nb_4 (6, 8)
nb_5 (1, 8)
nb_6 (12, 8)
nb_7 (23, 8)
New_HI_ID_LY_OP_SP_clean_clean (89, 8)
ob (6, 8)
ob_2 (32, 8)
ob_3 (61, 8)
ob_4 (34, 8)
oi (1, 8)
oi_2 (80, 8)
oi_3 (35, 8)
oo (4, 8)
oo_2 (14, 8)
oo_3 (0, 8)
oo_4 (67, 8)
oo_5 (3, 8)
oo_6 (45, 8)
os (3, 8)
os_2 (106, 8)
Persian_data (1100, 8)
Persian_data3_clean (99, 8)
ra (0, 8)
ra_2 (10, 8)
ra_3 (44, 8)
ra_4 (76, 8)
re (92, 8)
re_2 (31, 8)
re_3 (36, 8)
re_4 (28, 8)
rs (151, 8)
rv (9, 8)
rv_2 (10, 8)
rv_3 (53, 8)
rv_4 (28, 8)
rv_5 (54, 8)
SP_clean_dedup (160, 8)
sr (97, 8)
sr_2 (95, 8)
sr_3 (27,

# Remove NSFW/HATE from data

In [5]:
search_term = "NSFW"

results = []

for dataset_name, df in dfs_noNA.items():
    matches = df[df["Tags"].astype(str).str.contains(search_term, case=False, na=False)].copy()

    if not matches.empty:
        matches["Dataset"] = dataset_name
        results.append(matches)

# Combine all matching rows into one DataFrame
if results:
    results_df = pd.concat(results, ignore_index=True)
   #print(results_df)
else:
    print("No matches found.")

In [6]:
search_terms = ["HATE", "NSFW"]

pattern = "|".join(search_terms) 

dfs_filtered = {}

for dataset_name, df in dfs_noNA.items():
    dfs_filtered[dataset_name] = df[
        ~df["Tags"].astype(str).str.contains(pattern, case=False, na=False)
    ].copy()

In [7]:
for dataset_name, df in dfs_filtered.items():
    print(f"{dataset_name}: {df.shape}")

ed: (47, 8)
ed_2: (98, 8)
ed_3: (5, 8)
en: (3, 8)
en_10: (17, 8)
en_11: (19, 8)
en_2: (3, 8)
en_3: (11, 8)
en_4: (8, 8)
en_5: (7, 8)
en_6: (16, 8)
en_7: (4, 8)
en_8: (4, 8)
en_9: (24, 8)
fi: (147, 8)
HI_2_clean_dedup: (85, 8)
HI_clean_dedup: (61, 8)
HI_ID_LY_SP_clean2_dedup: (6, 8)
ID_clean_dedup: (225, 8)
it: (6, 8)
it_2: (7, 8)
it_3: (91, 8)
lt: (149, 8)
lt_2: (58, 8)
lt_3: (75, 8)
lt_4: (30, 8)
LY_clean_dedup: (181, 8)
MT: (148, 8)
nb: (14, 8)
nb_2: (98, 8)
nb_3: (0, 8)
nb_4: (6, 8)
nb_5: (1, 8)
nb_6: (12, 8)
nb_7: (23, 8)
New_HI_ID_LY_OP_SP_clean_clean: (89, 8)
ob: (6, 8)
ob_2: (32, 8)
ob_3: (59, 8)
ob_4: (33, 8)
oi: (1, 8)
oi_2: (80, 8)
oi_3: (35, 8)
oo: (4, 8)
oo_2: (14, 8)
oo_3: (0, 8)
oo_4: (67, 8)
oo_5: (3, 8)
oo_6: (44, 8)
os: (3, 8)
os_2: (106, 8)
Persian_data: (1100, 8)
Persian_data3_clean: (99, 8)
ra: (0, 8)
ra_2: (10, 8)
ra_3: (44, 8)
ra_4: (76, 8)
re: (92, 8)
re_2: (31, 8)
re_3: (36, 8)
re_4: (28, 8)
rs: (150, 8)
rv: (9, 8)
rv_2: (10, 8)
rv_3: (53, 8)
rv_4: (28, 8)
rv_5:

In [8]:
#Sanity check
search_terms = ["HATE", "NSFW"]
pattern = "|".join(search_terms)

for dataset_name, df in dfs_filtered.items():
    n_matches = df["Tags"].astype(str).str.contains(
        pattern, case=False, na=False
    ).sum()

    print(f"{dataset_name}: {n_matches} matches remaining")

ed: 0 matches remaining
ed_2: 0 matches remaining
ed_3: 0 matches remaining
en: 0 matches remaining
en_10: 0 matches remaining
en_11: 0 matches remaining
en_2: 0 matches remaining
en_3: 0 matches remaining
en_4: 0 matches remaining
en_5: 0 matches remaining
en_6: 0 matches remaining
en_7: 0 matches remaining
en_8: 0 matches remaining
en_9: 0 matches remaining
fi: 0 matches remaining
HI_2_clean_dedup: 0 matches remaining
HI_clean_dedup: 0 matches remaining
HI_ID_LY_SP_clean2_dedup: 0 matches remaining
ID_clean_dedup: 0 matches remaining
it: 0 matches remaining
it_2: 0 matches remaining
it_3: 0 matches remaining
lt: 0 matches remaining
lt_2: 0 matches remaining
lt_3: 0 matches remaining
lt_4: 0 matches remaining
LY_clean_dedup: 0 matches remaining
MT: 0 matches remaining
nb: 0 matches remaining
nb_2: 0 matches remaining
nb_3: 0 matches remaining
nb_4: 0 matches remaining
nb_5: 0 matches remaining
nb_6: 0 matches remaining
nb_7: 0 matches remaining
New_HI_ID_LY_OP_SP_clean_clean: 0 matche

In [9]:
dfs_filtered["sr_3"].head()

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register
93,https://media.salameno.ir/tag/%D8%B3%D8%A7%D8%...,c977ad4d006a638cf010b0518c95e97c,2021-09-28T02:15:19Z,252541\n-\nمرحله نیمه نهایی و فینال جام حذفی ر...,NA ; NA,sr ; ne,None,"{""MT"": 0.07, ""LY"": 0.059000000000000004, ""SP"":..."
98,https://baztab.ir/%D8%AA%DB%8C%D9%85%E2%80%8C%...,2ac7f27fb9a86cdbaabbc13ba4fe6c35,2021-09-29T01:16:00Z,فدراسیون کشتی به تیمهای سوم رقابتهای لیگ کشتی ...,NA,sr,None,"{""MT"": 0.065, ""LY"": 0.052000000000000005, ""SP""..."
103,http://mehranamini.mihanblog.com/post/archive/...,5e8d7e711ae0977ce62caf30143f2746,2014-08-03T17:10:12Z,- نقاشی انلاین\n- جذب نویسند و مدیر فعال\n- ۱۰...,-,-,MLPT,"{""MT"": 0.045, ""LY"": 0.069, ""SP"": 0.092, ""ID"": ..."
105,http://iranerooz.com/tags/%D8%A7%D8%B2%20%D9%8...,c472c21606ce3566ec3bd72340dd4a6d,2014-08-01T13:03:15Z,- اشکان دژاگه با مشکل جدیدی روبرو شد ! تلفظ اس...,NA ; NA,sr ; ne,OTHLI,"{""MT"": 0.078, ""LY"": 0.064, ""SP"": 0.09, ""ID"": 0..."
109,http://s-7-1-5-9-8-s.blogfa.com/1390/12/1,d89a6ee5f73d93e5a077143424d6a99f,2014-08-02T15:38:58Z,فرهاد مجیدی کاپیتان سابق استقلال پاش بد جور به...,NA ; OP,sr ; ob,None,"{""MT"": 0.056, ""LY"": 0.07200000000000001, ""SP"":..."


# Remove Duplicate

In [24]:
seen_ids = set()
dfs_dedup = {}

for dataset_name, df in dfs_filtered.items():

    # Keep only rows whose id has not been seen before
    mask = ~df["id"].isin(seen_ids)
    dfs_dedup[dataset_name] = df[mask].copy()

    # Add the ids we kept to the seen set
    seen_ids.update(dfs_dedup[dataset_name]["id"])

In [25]:
for dataset_name in dfs_filtered:
    before = len(dfs_filtered[dataset_name])
    after = len(dfs_dedup[dataset_name])

    print(f"{dataset_name}: {before} → {after} (removed {before-after})")

ed: 47 → 47 (removed 0)
ed_2: 98 → 98 (removed 0)
ed_3: 5 → 5 (removed 0)
en: 3 → 3 (removed 0)
en_10: 17 → 17 (removed 0)
en_11: 19 → 19 (removed 0)
en_2: 3 → 3 (removed 0)
en_3: 11 → 11 (removed 0)
en_4: 8 → 8 (removed 0)
en_5: 7 → 7 (removed 0)
en_6: 16 → 16 (removed 0)
en_7: 4 → 4 (removed 0)
en_8: 4 → 4 (removed 0)
en_9: 24 → 24 (removed 0)
fi: 147 → 147 (removed 0)
HI_2_clean_dedup: 85 → 85 (removed 0)
HI_clean_dedup: 61 → 61 (removed 0)
HI_ID_LY_SP_clean2_dedup: 6 → 6 (removed 0)
ID_clean_dedup: 225 → 225 (removed 0)
it: 6 → 6 (removed 0)
it_2: 7 → 7 (removed 0)
it_3: 91 → 91 (removed 0)
lt: 149 → 149 (removed 0)
lt_2: 58 → 58 (removed 0)
lt_3: 75 → 75 (removed 0)
lt_4: 30 → 30 (removed 0)
LY_clean_dedup: 181 → 181 (removed 0)
MT: 148 → 147 (removed 1)
nb: 14 → 13 (removed 1)
nb_2: 98 → 93 (removed 5)
nb_3: 0 → 0 (removed 0)
nb_4: 6 → 5 (removed 1)
nb_5: 1 → 1 (removed 0)
nb_6: 12 → 12 (removed 0)
nb_7: 23 → 23 (removed 0)
New_HI_ID_LY_OP_SP_clean_clean: 89 → 89 (removed 0)
ob: 

In [ ]:
#Sanity Check
all_df = pd.concat(dfs_dedup.values(), ignore_index=True)

print("Duplicate IDs remaining:", all_df["id"].duplicated().sum())

Duplicate IDs remaining: 0


# Junk/no Junk dataset

In [27]:
import numpy as np

for dataset_name, df in dfs_dedup.items():

    conditions = [
        (df["Turku_NLP"] == "-") & (df["Turku_NLP_sub"] == "-"),
        (df["Turku_NLP"] == "MT")
    ]

    choices = [0, 0]

    df["Binary"] = np.select(conditions, choices, default=1)

In [28]:
dfs_dedup["sr_3"][["Turku_NLP", "Turku_NLP_sub", "Tags" , "Binary"]].head()

,Turku_NLP,Turku_NLP_sub,Tags,Binary
93,NA ; NA,sr ; ne,None,1
98,NA,sr,None,1
103,-,-,MLPT,0
105,NA ; NA,sr ; ne,OTHLI,1
109,NA ; OP,sr ; ob,None,1


In [29]:
dfs_dedup["Persian_data"][["Turku_NLP", "Turku_NLP_sub", "Tags" , "Binary"]].tail()

,Turku_NLP,Turku_NLP_sub,Tags,Binary
1095,IP ; OP,ds ; rv,OTHLI,1
1096,-,-,JNK,0
1097,OP,av,OTHLI,1
1098,NA ; OP,on ; oo,None,1
1099,OP ; ID,ob ; -,None,1


In [39]:
combined_binary_df = pd.concat(
    [
        df.assign(source=name)
        for name, df in dfs_dedup.items()
    ],
    ignore_index=True
)

## CSV save

In [41]:
combined_binary_df.to_csv('binary_dataset.CSV', index=False)

# Hugging Face Dataset

In [42]:
from datasets import Dataset

dataset_binary= Dataset.from_pandas(combined_binary_df, preserve_index=False)

c:\Users\alrazz\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
from datasets import ClassLabel

dataset_binary = dataset_binary.cast_column(
    "Binary",
    ClassLabel(num_classes=2, names=["0", "1"])
)

Casting the dataset: 100%|██████████| 4441/4441 [00:00<00:00, 32351.91 examples/s]


In [46]:
#80-10-10
train_test = dataset_binary.train_test_split(
    test_size=0.2,
    stratify_by_column="Binary",
    seed=66
)

valid_test = train_test["test"].train_test_split(
    test_size=0.5,
    stratify_by_column="Binary",
    seed=66
)

train_ds = train_test["train"]
valid_ds = valid_test["train"]
test_ds = valid_test["test"]

In [47]:
from collections import Counter

print(Counter(train_ds["Binary"]))
print(Counter(valid_ds["Binary"]))
print(Counter(test_ds["Binary"]))

Counter({1: 3138, 0: 414})
Counter({1: 393, 0: 51})
Counter({1: 393, 0: 52})


In [48]:
from datasets import DatasetDict

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})

dataset_dict.save_to_disk("binary_dataset")

#from datasets import load_from_disk

#dataset = load_from_disk("binary_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 445/445 [00:00<00:00, 5208.35 examples/s]


# Registers

In [49]:
def process_turku(df, main_col="Turku_NLP", sub_col="Turku_NLP_sub"):
    """
    Processes a dataframe by splitting the given Turku columns,
    pairing elementwise, exploding, and fixing '-' entries.
    """

    df_copy = df.copy()

    # Step 1: Split into lists
    df_copy[f"{main_col}_split"] = df_copy[main_col].str.split(" ; ")
    df_copy[f"{sub_col}_split"] = df_copy[sub_col].str.split(" ; ")

    # Step 2: Pair elementwise
    df_copy["paired"] = df_copy.apply(
        lambda x: list(zip(x[f"{main_col}_split"], x[f"{sub_col}_split"])),
        axis=1
    )

    # Step 3: Explode
    exploded = df_copy.explode("paired").copy()

    # Step 4: Restore into two separate columns
    exploded[f"{main_col}_split"] = exploded["paired"].apply(
        lambda x: x[0] if pd.notna(x) else None
    )
    exploded[f"{sub_col}_split"] = exploded["paired"].apply(
        lambda x: x[1] if pd.notna(x) else None
    )

    exploded = exploded.drop(columns=["paired"])

    # Step 5: Replace "-" in sub column
    exploded[f"{sub_col}_split_modified"] = exploded[f"{sub_col}_split"]

    mask = (
        (exploded[f"{sub_col}_split"] == "-")
        & (exploded[f"{main_col}_split"] != "-")
    )

    exploded.loc[mask, f"{sub_col}_split_modified"] = exploded.loc[
        mask, f"{main_col}_split"
    ]

    return exploded

In [50]:
dfs_processed = {}

for name, df in dfs_dedup.items():
    if df.empty:
        continue

    processed = process_turku(df)
    processed["source"] = name
    dfs_processed[name] = processed

df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [51]:
df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [52]:
print(df_all["Turku_NLP_sub_split_modified"].unique())
print(df_all["Turku_NLP_split"].unique())

['ed' 'ID' 'on' 'ob' 'os' 'ra' 'rs' 'dtp' 'oo' 'ne' 'LY' 'nb' 'oe' 'MT'
 'oi' 'lt' 'en' 'fi' 'av' 'it' 'oh' 'ds' 're' 'rv' '-' 'sr']
['IP' 'ID' 'NA' 'OP' 'SP' 'IN' 'LY' 'MT' 'HI' '-']


In [53]:
print(len((df_all["Turku_NLP_sub_split_modified"].unique()))) #should be 26
print(len(df_all["Turku_NLP_split"].unique())) #should be 10

26
10


In [54]:
# Find erros like lt ;oh (where space after ; forgotten)
def find_value(dfs_processed, column, value):
    for name, df in dfs_processed.items():
        matches = df[df[column] == value]

        if not matches.empty:
            print(f"\n===== {name} ({len(matches)} matches) =====")
            display(matches)

In [55]:
find_value(
    dfs_processed,
    "Turku_NLP_sub_split_modified", #Turku_NLP_split
    "op" #for example lt ;oh
)

In [56]:
df_all.head()
#Turku_NLP_split is the main register
#Turku_NLP_sub_split is the sub register
#Turku_NLP_sub_split_modified for subregister "-" --> replaced by main register

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register,Binary,Turku_NLP_split,Turku_NLP_sub_split,Turku_NLP_sub_split_modified,source
0,http://roshangari.info/?p=35434,110ae9e775485f536c4f1cdea087cbd7,2020-01-25T19:50:48Z,e.shafagh@yahoo.com\n“تاریخ گواهی خواهد داد که...,IP ; ID,ed ; -,OTHLI,"{""MT"": 0.074, ""LY"": 0.084, ""SP"": 0.105, ""ID"": ...",1,IP,ed,ed,ed
1,http://roshangari.info/?p=35434,110ae9e775485f536c4f1cdea087cbd7,2020-01-25T19:50:48Z,e.shafagh@yahoo.com\n“تاریخ گواهی خواهد داد که...,IP ; ID,ed ; -,OTHLI,"{""MT"": 0.074, ""LY"": 0.084, ""SP"": 0.105, ""ID"": ...",1,ID,-,ID,ed
2,https://www.tinn.ir/%D8%A8%D8%AE%D8%B4-%D9%88%...,32962503dcdb08ee1bc4d2e9d5240bc7,2020-01-19T18:40:03Z,در کشورهای پیشرفته که حملونقل بر اساس پژوهشهای...,IP,ed,None,"{""MT"": 0.089, ""LY"": 0.085, ""SP"": 0.125, ""ID"": ...",1,IP,ed,ed,ed
3,http://p313.ir/post-124289.html,0e6562306fcb8db55e019445407cb487,2016-10-26T00:51:16Z,مهدی محمدی طی یادداشتی در روزنامه وطن امروز نو...,IP,ed,None,"{""MT"": 0.093, ""LY"": 0.063, ""SP"": 0.11900000000...",1,IP,ed,ed,ed
4,http://shakhesnews.com/%db%b6-%d8%af%d8%a7%d9%...,588d3acafdc57f5122849f3163c1d357,2016-10-27T18:45:54Z,شاخص : اگر مسیر مذاکرات در همین جهت ادامه یابد...,IP,ed,OTHLI,"{""MT"": 0.092, ""LY"": 0.07100000000000001, ""SP"":...",1,IP,ed,ed,ed


In [57]:
df_all.nunique()

u                               4441
id                              4441
ts                              4438
text                            4441
Turku_NLP                        190
Turku_NLP_sub                    472
Tags                               8
web-register                    4384
Binary                             2
Turku_NLP_split                   10
Turku_NLP_sub_split               23
Turku_NLP_sub_split_modified      26
source                            68
dtype: int64

## Check Inconsistent IDs

In [58]:
# Columns you want to compare
cols_to_check = ["Turku_NLP", "Turku_NLP_sub"]

# Group by ID
grouped = df_all.groupby("id")

inconsistent_ids = []

for doc_id, group in grouped:
    # For each of the columns, check if there is more than one unique non-null value
    inconsistent = False
    for col in cols_to_check:
        unique_vals = group[col].dropna().unique()
        if len(unique_vals) > 1:
            inconsistent = True
    if inconsistent:
        inconsistent_ids.append(doc_id)

print("Inconsistent IDs:", inconsistent_ids)
print(f"Total inconsistent IDs: {len(inconsistent_ids)}")

# Optional: print detailed info like Claude did
for doc_id in inconsistent_ids:
    print("\nID", doc_id, "has inconsistent values:")
    display(df_all[df_all["id"] == doc_id][["id", "Turku_NLP", "Turku_NLP_sub", "source"]])


Inconsistent IDs: []
Total inconsistent IDs: 0


## CSV

In [59]:
df_all.to_csv('Register_explode.CSV', index=False)

## Tableau